# Using pretrained models (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [2]:
# !pip install datasets evaluate transformers[sentencepiece]

We select the `camembert-base` checkpoint to try it out. The identifier `camembert-base` is all we need to start using it! As we have seen in previous chapters, we can instantiate it using the `pipeline()` function:

In [ ]:
from transformers import pipeline

camembert_fill_mask = pipeline("fill-mask", model="camembert-base")
results = camembert_fill_mask("Le camembert est <mask> :)")

In [4]:
from pprint import pprint

pprint(results)

[{'score': 0.4909065365791321,
  'sequence': 'Le camembert est délicieux :)',
  'token': 7200,
  'token_str': 'délicieux'},
 {'score': 0.10556869208812714,
  'sequence': 'Le camembert est excellent :)',
  'token': 2183,
  'token_str': 'excellent'},
 {'score': 0.034532953053712845,
  'sequence': 'Le camembert est succulent :)',
  'token': 26202,
  'token_str': 'succulent'},
 {'score': 0.033031269907951355,
  'sequence': 'Le camembert est meilleur :)',
  'token': 528,
  'token_str': 'meilleur'},
 {'score': 0.0300764013081789,
  'sequence': 'Le camembert est parfait :)',
  'token': 1654,
  'token_str': 'parfait'}]


We can also instantiate the checkpoint using the model architecture directly:

In [ ]:
from transformers import CamembertTokenizer, CamembertForMaskedLM

tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
model = CamembertForMaskedLM.from_pretrained("camembert-base")

However, we recommend using the `Auto* classes` instead, as these are by design architecture-agnostic. While the previous code sample limits users to checkpoints loadable in the CamemBERT architecture, using the `Auto*` classes makes switching checkpoints simple:

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("camembert-base")
model = AutoModelForMaskedLM.from_pretrained("camembert-base")

In [ ]:
# Cleaning up
import gc
import torch

del camembert_fill_mask
del tokenizer
del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Cleanup

When I am finished experimenting with a downloaded model, I can clean up its
resources to free **RAM, GPU VRAM, and disk space**.

The cleanup process has two parts:

1. **Free memory:** Delete the model, tokenizer, and pipeline objects, run Python's
   garbage collector, and clear PyTorch's unused GPU cache.
2. **Free disk space:** Find the Hugging Face cache location, search for cached
   CamemBERT files, review the matching directories, and then remove only the
   CamemBERT-related cache and lock files.

The search step is useful because it lets me see exactly where the downloaded
model files are stored before deleting them.

> **Note:** Removing a model from the Hugging Face cache does not uninstall
> Transformers or PyTorch. It only removes the downloaded model files. If I use
> the model again later, Hugging Face will download it again.

In [14]:
import gc
import os
import shutil
from pathlib import Path

import torch

# 1. Free RAM and GPU VRAM


for name in ["camembert_fill_mask", "tokenizer", "model"]:
    if name in globals():
        del globals()[name]

# Run Python's garbage collector
gc.collect()

# Clear unused PyTorch GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
# 2. Find the Hugging Face cache

cache_dir = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))

hub_dir = cache_dir / "hub"

print("=" * 60)
print("Hugging Face cache")
print("=" * 60)
print(cache_dir)

In [ ]:
# 3. Search for CamemBERT-related cached files

print("\n" + "=" * 60)
print("Searching for CamemBERT cache files...")
print("=" * 60)

camembert_matches = list(hub_dir.rglob("*camembert*"))

if camembert_matches:
    for path in camembert_matches:
        print(path)
else:
    print("No CamemBERT cache files found.")

In [ ]:
# 4. List only CamemBERT cache directories

camembert_dirs = [
    hub_dir / "models--camembert--camembert-base-wikipedia-4gb",
    hub_dir / "models--camembert-base",
    # Corresponding lock directories
    hub_dir / ".locks" / "models--camembert--camembert-base-wikipedia-4gb",
    hub_dir / ".locks" / "models--camembert-base",
]


# 5. Show what will be deleted


print("\n" + "=" * 60)
print("CamemBERT directories selected for deletion")
print("=" * 60)

for path in camembert_dirs:
    if path.exists():
        print(path)
    else:
        print(f"Not found: {path}")

In [ ]:
# 6. Delete the selected CamemBERT cache


print("\n" + "=" * 60)
print("Deleting CamemBERT cache...")
print("=" * 60)

for path in camembert_dirs:
    if path.exists():
        print(f"Deleting: {path}")
        shutil.rmtree(path)

print("\nCleanup complete.")

In [19]:
# 7. Verify that CamemBERT cache files are gone

print("\n" + "=" * 60)
print("Verifying cleanup...")
print("=" * 60)

remaining = list(hub_dir.rglob("*camembert*"))

if remaining:
    print("The following CamemBERT files still remain:")
    for path in remaining:
        print(path)
else:
    print("No CamemBERT cache files found.")


Verifying cleanup...
No CamemBERT cache files found.
